# Notebook 03: Jump Hyperparameter Tuning

This notebook performs a multi-objective grid search over the jump parameters (ε, λ)
to find the values that best reproduce volatility clustering while preserving distributional fidelity.

### Objective Function (Eq. 3 from the paper)
$$J(\epsilon, \lambda) = \sum_{\tau=1}^{L} [\text{ACF}_{obs}(\tau) - \text{ACF}_{sim}(\tau)]^2 + w_K [K_{obs} - K_{sim}]^2$$

where ACF is computed on |Gₜ|, L=252, and wK=0.20.

### Outputs
- **Figure S2 equivalent**: Grid search heatmap + optimal ACF comparison
- Saved optimal (ε*, λ*) for downstream notebooks

## Setup

In [ ]:
include("../Include.jl");

In [ ]:
# --- CONSTANTS ---
ticker = "SPY";
number_of_states = 13;
L = 252;                    # ACF max lag
n_paths_grid = 200;         # paths per grid point
w_K = 0.20;                 # kurtosis penalty weight

# Grid ranges (adapted from paper's Algorithm 4)
ε_grid = [1e-4, 2.5e-4, 5e-4, 1e-3, 2.5e-3, 5e-3, 1e-2, 2.5e-2];
λ_grid = [10, 25, 40, 55, 70, 85, 100, 130, 160];

## Load Pre-Trained Base Model

Load the model trained in Notebook 02, or train a fresh one.

In [ ]:
# --- LOAD OR TRAIN BASE MODEL ---
model_path = joinpath(_PATH_TO_DATA, "CHMM-$(ticker)-K$(number_of_states)-base.jld2");

if isfile(model_path)
    saved = load(model_path);
    model = saved["model"];
    T_matrix = saved["T_matrix"];
    π_stationary = saved["pi_stationary"];
    in_sample_data = saved["in_sample_data"];
    println("Loaded pre-trained model from disk.")
else
    # Train fresh
    risk_free_rate = 0.0; Δt = 1/252;
    train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
    maximum_number_trading_days = nrow(train_dataset["AAPL"]);
    dataset = Dict{String,DataFrame}();
    for (t, data) ∈ train_dataset
        if nrow(data) == maximum_number_trading_days
            dataset[t] = data;
        end
    end
    list_of_all_tickers = keys(dataset) |> collect |> sort;
    all_R = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
    Rᵢ = findfirst(x -> x == ticker, list_of_all_tickers) |> i -> all_R[:, i];
    in_sample_data = Rᵢ[1:(maximum_number_trading_days-1)];
    
    model = build(MyContinuousHiddenMarkovModel, (
        observations = in_sample_data, number_of_states = number_of_states, max_iter = 60));
    
    K = length(model.states);
    T_matrix = zeros(K, K);
    for i in 1:K; T_matrix[i, :] = probs(model.transition[i]); end
    π_stationary = (T_matrix^1000)[1, :];
    println("Trained fresh model.")
end

K = length(model.states);
number_of_steps = length(in_sample_data);
println("K=$K states, T=$(number_of_steps) steps")

## Compute Observed Targets

In [ ]:
# --- OBSERVED TARGETS ---
target_acf = autocor(abs.(in_sample_data), 1:L);
target_kurtosis = sum(((in_sample_data .- mean(in_sample_data)) ./ std(in_sample_data)).^4) / length(in_sample_data) - 3.0;

println("Observed |Gₜ| mean ACF: $(round(mean(target_acf), digits=4))")
println("Observed excess kurtosis: $(round(target_kurtosis, digits=2))")

## Grid Search

In [ ]:
# --- GRID SEARCH (Algorithm 4) ---
start_distribution = Categorical(π_stationary);
objective_matrix = zeros(length(ε_grid), length(λ_grid));

best_J = Inf;
best_ε = ε_grid[1];
best_λ = λ_grid[1];

println("Running grid search: $(length(ε_grid)) x $(length(λ_grid)) = $(length(ε_grid)*length(λ_grid)) grid points...")

for (ei, ε) in enumerate(ε_grid)
    for (li, λ) in enumerate(λ_grid)
        
        # Build jump model for this (ε, λ)
        jm = build(MyContinuousHiddenMarkovModelWithJumps, (
            base_model = model, epsilon = ε, lambda = Float64(λ)));
        
        # Simulate n_paths_grid paths and compute ensemble metrics
        acf_sum = zeros(L);
        kurt_sum = 0.0;
        
        for p in 1:n_paths_grid
            start_state = rand(start_distribution);
            states = jm(start_state, number_of_steps);
            returns = [rand(jm.emission[s]) for s in states];
            
            acf_sum .+= autocor(abs.(returns), 1:L);
            μ_r = mean(returns); σ_r = std(returns);
            kurt_sum += sum(((returns .- μ_r) ./ σ_r).^4) / length(returns) - 3.0;
        end
        
        acf_sim = acf_sum ./ n_paths_grid;
        kurt_sim = kurt_sum / n_paths_grid;
        
        # Objective: ACF SSE + kurtosis penalty
        J = sum((target_acf .- acf_sim).^2) + w_K * (target_kurtosis - kurt_sim)^2;
        objective_matrix[ei, li] = J;
        
        if J < best_J
            best_J = J;
            best_ε = ε;
            best_λ = λ;
        end
    end
    println("  ε = $ε done.")
end

println("\nOptimal: ε* = $best_ε, λ* = $best_λ, J* = $(round(best_J, digits=4))")

## Figure S2: Grid Search Heatmap

In [ ]:
# --- FIGURE S2a: HEATMAP ---
ε_labels = [string(round(e, sigdigits=2)) for e in ε_grid];
λ_labels = [string(Int(l)) for l in λ_grid];

p_heat = heatmap(λ_labels, ε_labels, log10.(objective_matrix),
    title="Grid Search: J(ε, λ) in log₁₀ scale", titlefontsize=10,
    xlabel="λ (mean jump duration)", ylabel="ε (jump probability)",
    color=:viridis, size=(700, 500));

# Mark optimal point
opt_li = findfirst(x -> x == best_λ, λ_grid);
opt_ei = findfirst(x -> x == best_ε, ε_grid);
scatter!(p_heat, [opt_li], [opt_ei], ms=12, color=:red, markershape=:star5, label="Optimal");

display(p_heat)
savefig(p_heat, joinpath(_PATH_TO_FIGURES, "Fig-Grid-Search-$(ticker).svg"))

## Figure S2b: ACF at Optimal Parameters

In [ ]:
# --- FIGURE S2b: ACF AT OPTIMAL ---
# Generate 500 paths at optimal parameters and compute ACF envelope
n_validate = 500;
optimal_model = build(MyContinuousHiddenMarkovModelWithJumps, (
    base_model = model, epsilon = best_ε, lambda = Float64(best_λ)));

acf_archive = zeros(L, n_validate);
for p in 1:n_validate
    start_state = rand(start_distribution);
    states = optimal_model(start_state, number_of_steps);
    returns = [rand(optimal_model.emission[s]) for s in states];
    acf_archive[:, p] = autocor(abs.(returns), 1:L);
end

acf_mean = mean(acf_archive, dims=2)[:];
acf_p10 = [quantile(acf_archive[τ, :], 0.10) for τ in 1:L];
acf_p90 = [quantile(acf_archive[τ, :], 0.90) for τ in 1:L];

τ = 1:L;
p_acf = plot(τ, target_acf, lw=2, color=:red, ls=:dash, label="Observed",
    title="ACF(|Gₜ|) at Optimal (ε*=$(best_ε), λ*=$(best_λ))", titlefontsize=10,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p_acf, τ, acf_mean, lw=2, color=:navy, label="Simulated (mean)");
plot!(p_acf, τ, acf_p10, fillrange=acf_p90, alpha=0.2, color=:navy, label="10th-90th percentile");

ci = 2.576 / sqrt(number_of_steps);
plot!(p_acf, τ, ci .* ones(L), lw=1, color=:black, ls=:dot, label="95% sig. band");

display(p_acf)
savefig(p_acf, joinpath(_PATH_TO_FIGURES, "Fig-ACF-Optimal-$(ticker).svg"))

## Save Optimal Parameters

In [ ]:
# --- SAVE ---
path_to_save = joinpath(_PATH_TO_DATA, "CHMM-$(ticker)-K$(number_of_states)-tuned.jld2");
save(path_to_save, Dict(
    "model" => model,
    "jump_model" => optimal_model,
    "T_matrix" => T_matrix,
    "pi_stationary" => π_stationary,
    "in_sample_data" => in_sample_data,
    "best_epsilon" => best_ε,
    "best_lambda" => best_λ,
    "objective_matrix" => objective_matrix,
    "number_of_states" => number_of_states
));
println("Saved tuned model to: $path_to_save")

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.